# FDIR Scenario Comparison

This notebook compares nominal and faulted runs, then inspects the event log to see how detection and recovery unfold over time.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(repo_root / 'src'))

from orbital_guardian.simulation import run_scenario


In [ ]:
nominal = run_scenario(repo_root / 'configs' / 'nominal.json')
faulted = run_scenario(repo_root / 'configs' / 'fdir_demo.json')

comparison = {
    'nominal': nominal.summary,
    'faulted': faulted.summary,
}
comparison


In [ ]:
nominal_soc = [sample.battery_soc_pct for sample in nominal.telemetry]
faulted_soc = [sample.battery_soc_pct for sample in faulted.telemetry]
nominal_time = [sample.time_minute for sample in nominal.telemetry]
faulted_time = [sample.time_minute for sample in faulted.telemetry]

try:
    import matplotlib.pyplot as plt
except ImportError:
    print('Install matplotlib to render plots from this notebook.')
else:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)
    axes[0].plot(nominal_time, nominal_soc, label='Nominal', color='#1b4d8c', linewidth=2)
    axes[0].plot(faulted_time, faulted_soc, label='Faulted', color='#c1121f', linewidth=2)
    axes[0].set_ylabel('Battery SoC [%]')
    axes[0].legend(loc='upper right')
    axes[0].grid(alpha=0.3)
    axes[1].step(
        faulted_time,
        [len(sample.alerts) for sample in faulted.telemetry],
        where='post',
        color='#f77f00',
        linewidth=2,
    )
    axes[1].set_ylabel('Faulted alert count')
    axes[1].set_xlabel('Mission elapsed time [min]')
    axes[1].grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


In [ ]:
[
    {
        'time_minute': event.time_minute,
        'category': event.category,
        'message': event.message,
        'metadata': event.metadata,
    }
    for event in faulted.events[:20]
]
